# Raw video VideoMAE local experiment - 50 per label

?? 10? ?? ??? ??? ?? VideoMAE ??? ????.

?? ??:
- ?? ?? ??? 50?
- ? 200? ?? ??
- train/val/test = 70/20/10
- frame_count=16

??:
1. freeze_backbone=True, lr=1e-4
2. freeze_backbone=False, lr=1e-5


In [8]:
from pathlib import Path
import csv
import json
import os
import subprocess
import sys
from collections import Counter, defaultdict

RUN_INSTALL_REQUIREMENTS = False
RUN_DOWNLOAD = True
RUN_TRAIN_EXP1 = True
RUN_TRAIN_EXP2 = True
RUN_TRAIN_EXP3 = True

PROJECT_ROOT = Path(r'D:\dev\SKN27-FINAL-3Team')
if not PROJECT_ROOT.exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'ai').exists() and (candidate / 'etl').exists() and (candidate / 'storage').exists():
            PROJECT_ROOT = candidate
            break

MANIFEST_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/manifests'
RAW_VIDEO_DIR = PROJECT_ROOT / 'storage/vision/datasets/classification/raw_videos'
MODEL_DIR = PROJECT_ROOT / 'storage/vision/models/videomae_raw_video'
SAMPLE_MANIFEST = MANIFEST_DIR / 'sample_700_coarse_manifest.csv'
FULL_DOWNLOAD_MANIFEST = MANIFEST_DIR / 'train_700_download_manifest.csv'
MANIFEST_50 = MANIFEST_DIR / 'train_50_raw_video_manifest.csv'
SPLIT_MANIFEST_50 = MANIFEST_DIR / 'train_50_raw_video_manifest_split.csv'

FRAME_COUNT = 16
BATCH_SIZE = 1
EPOCHS = 5
SEED = 42
DEVICE = 'auto'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FULL_DOWNLOAD_MANIFEST:', FULL_DOWNLOAD_MANIFEST)
print('SPLIT_MANIFEST_50:', SPLIT_MANIFEST_50)


PROJECT_ROOT: D:\dev\SKN27-FINAL-3Team
FULL_DOWNLOAD_MANIFEST: D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_700_download_manifest.csv
SPLIT_MANIFEST_50: D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv


In [9]:
def run_command(command, *, enabled=True, timeout=None):
    command = list(map(str, command))
    print('$', ' '.join(command), flush=True)
    if not enabled:
        print('SKIPPED')
        return None
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        encoding='utf-8',
        errors='replace',
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        env=env,
    )
    try:
        for line in process.stdout:
            print(line, end='')
        returncode = process.wait(timeout=timeout)
    except Exception:
        process.kill()
        raise
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)
    return returncode


def read_csv(path):
    with Path(path).open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def write_csv(rows, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(key for row in rows for key in row.keys()))
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def resolve_local_video_path(path_value):
    if not path_value:
        return None
    path_text = str(path_value).replace('\\', '/')
    runpod_prefix = '/workspace/SKN27-FINAL-3Team/'
    if path_text.startswith(runpod_prefix):
        return PROJECT_ROOT / path_text[len(runpod_prefix):]
    path = Path(path_value)
    return path if path.is_absolute() else PROJECT_ROOT / path


def make_subset_manifest(source_manifest, output_manifest, per_label):
    rows = read_csv(source_manifest)
    selected = []
    counts = defaultdict(int)
    for row in rows:
        label = row.get('coarse_label') or row.get('label')
        if not label or counts[label] >= per_label:
            continue
        path = resolve_local_video_path(row.get('local_path') or row.get('file_path'))
        if path is not None:
            if not path.exists():
                continue
            row = dict(row)
            row['local_path'] = str(path)
            row['file_exists'] = 'True'
        selected.append(row)
        counts[label] += 1
    write_csv(selected, output_manifest)
    print('subset_manifest:', output_manifest)
    print('rows:', len(selected))
    print('label_counts:', dict(Counter(row.get('coarse_label') for row in selected)))
    return output_manifest


def make_split_manifest(source_manifest, output_manifest, train_ratio=0.7, val_ratio=0.2):
    rows = read_csv(source_manifest)
    grouped = defaultdict(list)
    for row in rows:
        grouped[row.get('coarse_label')].append(row)
    split_rows = []
    for label, label_rows in grouped.items():
        total = len(label_rows)
        train_end = int(total * train_ratio)
        val_end = train_end + int(total * val_ratio)
        for index, row in enumerate(label_rows):
            copied = dict(row)
            if index < train_end:
                copied['split'] = 'train'
            elif index < val_end:
                copied['split'] = 'val'
            else:
                copied['split'] = 'test'
            split_rows.append(copied)
    write_csv(split_rows, output_manifest)
    print('split_manifest:', output_manifest)
    print('rows:', len(split_rows))
    print('label_counts:', dict(Counter(row.get('coarse_label') for row in split_rows)))
    print('split_counts:', dict(Counter(row.get('split') for row in split_rows)))
    print('label_split_counts:')
    for key, value in sorted(Counter((row.get('coarse_label'), row.get('split')) for row in split_rows).items()):
        print(key, value)
    return output_manifest


def latest_run_dir(path):
    runs = [p for p in Path(path).glob('videomae_cls_*') if p.is_dir()]
    if not runs:
        raise FileNotFoundError(f'No runs found under {path}')
    return sorted(runs)[-1]


def build_train_command(experiment):
    command = [
        sys.executable,
        'ai/vision/train_videomae_classifier.py',
        '--manifest', experiment['manifest'],
        '--root-dir', PROJECT_ROOT,
        '--output-dir', experiment['output_dir'],
        '--label-column', 'coarse_label',
        '--frame-count', experiment['frame_count'],
        '--epochs', experiment['epochs'],
        '--batch-size', experiment['batch_size'],
        '--learning-rate', experiment['learning_rate'],
        '--weight-decay', experiment['weight_decay'],
        '--early-stopping-patience', experiment['early_stopping_patience'],
        '--seed', SEED,
        '--device', DEVICE,
        '--num-workers', '0',
        '--no-show-progress',
    ]
    if experiment['freeze_backbone']:
        command.append('--freeze-backbone')
    return command


def run_experiment(experiment, *, enabled=True):
    print('\n##', experiment['name'])
    print(json.dumps({k: str(v) for k, v in experiment.items()}, ensure_ascii=False, indent=2))
    run_command(build_train_command(experiment), enabled=enabled, timeout=None)
    if enabled:
        print('LAST_RUN_DIR:', latest_run_dir(experiment['output_dir']))


In [10]:
if not SAMPLE_MANIFEST.exists() and not FULL_DOWNLOAD_MANIFEST.exists():
    raise FileNotFoundError(f'Missing source manifest: {SAMPLE_MANIFEST}')

if not FULL_DOWNLOAD_MANIFEST.exists():
    run_command([
        sys.executable,
        'etl/vision/download_sampled_media.py',
        '--input', SAMPLE_MANIFEST,
        '--output', FULL_DOWNLOAD_MANIFEST,
        '--download-dir', RAW_VIDEO_DIR,
        '--label-column', 'coarse_label',
        '--per-label', '700',
        '--split', '',
    ], enabled=RUN_DOWNLOAD, timeout=None)

rows = read_csv(FULL_DOWNLOAD_MANIFEST)
print('download_rows:', len(rows))
print('label_counts:', dict(Counter(row.get('coarse_label') for row in rows)))
print('download_status:', dict(Counter(row.get('download_status') for row in rows)))
print('file_exists:', dict(Counter(row.get('file_exists') for row in rows)))


download_rows: 2800
label_counts: {'차대보행자': 700, '차대이륜차': 700, '차대자전거': 700, '차대차': 700}
download_status: {'exists': 921, 'downloaded': 1879}
file_exists: {'True': 2800}


In [11]:
make_subset_manifest(FULL_DOWNLOAD_MANIFEST, MANIFEST_50, per_label=50)
make_split_manifest(MANIFEST_50, SPLIT_MANIFEST_50)


subset_manifest: D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest.csv
rows: 200
label_counts: {'차대보행자': 50, '차대이륜차': 50, '차대자전거': 50, '차대차': 50}
split_manifest: D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv
rows: 200
label_counts: {'차대보행자': 50, '차대이륜차': 50, '차대자전거': 50, '차대차': 50}
split_counts: {'train': 140, 'val': 40, 'test': 20}
label_split_counts:
('차대보행자', 'test') 5
('차대보행자', 'train') 35
('차대보행자', 'val') 10
('차대이륜차', 'test') 5
('차대이륜차', 'train') 35
('차대이륜차', 'val') 10
('차대자전거', 'test') 5
('차대자전거', 'train') 35
('차대자전거', 'val') 10
('차대차', 'test') 5
('차대차', 'train') 35
('차대차', 'val') 10


WindowsPath('D:/dev/SKN27-FINAL-3Team/storage/vision/datasets/classification/manifests/train_50_raw_video_manifest_split.csv')

In [ ]:
EXPERIMENT_1 = {
    'name': 'exp1_raw50_freeze_lr1e-4_fc16',
    'manifest': SPLIT_MANIFEST_50,
    'output_dir': MODEL_DIR / 'per_label_50_exp1_freeze_lr1e-4',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.0001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': True,
}
run_experiment(EXPERIMENT_1, enabled=RUN_TRAIN_EXP1)



## exp1_raw50_freeze_lr1e-4_fc16
{
  "name": "exp1_raw50_freeze_lr1e-4_fc16",
  "manifest": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\datasets\\classification\\manifests\\train_50_raw_video_manifest_split.csv",
  "output_dir": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\models\\videomae_raw_video\\per_label_50_exp1_freeze_lr1e-4",
  "epochs": "5",
  "batch_size": "1",
  "learning_rate": "0.0001",
  "weight_decay": "0.05",
  "early_stopping_patience": "2",
  "frame_count": "16",
  "freeze_backbone": "True"
}
$ d:\dev\SKN27-FINAL-3Team\.venv\Scripts\python.exe ai/vision/train_videomae_classifier.py --manifest D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv --root-dir D:\dev\SKN27-FINAL-3Team --output-dir D:\dev\SKN27-FINAL-3Team\storage\vision\models\videomae_raw_video\per_label_50_exp1_freeze_lr1e-4 --label-column coarse_label --frame-count 16 --epochs 5 --batch-size 1 --learning-rate 0.0001 --weight-decay 0.05 --ear

In [ ]:
EXPERIMENT_2 = {
    'name': 'exp2_raw50_unfreeze_lr1e-5_fc16',
    'manifest': SPLIT_MANIFEST_50,
    'output_dir': MODEL_DIR / 'per_label_50_exp2_unfreeze_lr1e-5',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.00001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': False,
}
run_experiment(EXPERIMENT_2, enabled=RUN_TRAIN_EXP2)



## exp2_raw50_unfreeze_lr1e-5_fc16
{
  "name": "exp2_raw50_unfreeze_lr1e-5_fc16",
  "manifest": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\datasets\\classification\\manifests\\train_50_raw_video_manifest_split.csv",
  "output_dir": "D:\\dev\\SKN27-FINAL-3Team\\storage\\vision\\models\\videomae_raw_video\\per_label_50_exp2_unfreeze_lr1e-5",
  "epochs": "5",
  "batch_size": "1",
  "learning_rate": "1e-05",
  "weight_decay": "0.05",
  "early_stopping_patience": "2",
  "frame_count": "16",
  "freeze_backbone": "False"
}
$ d:\dev\SKN27-FINAL-3Team\.venv\Scripts\python.exe ai/vision/train_videomae_classifier.py --manifest D:\dev\SKN27-FINAL-3Team\storage\vision\datasets\classification\manifests\train_50_raw_video_manifest_split.csv --root-dir D:\dev\SKN27-FINAL-3Team --output-dir D:\dev\SKN27-FINAL-3Team\storage\vision\models\videomae_raw_video\per_label_50_exp2_unfreeze_lr1e-5 --label-column coarse_label --frame-count 16 --epochs 5 --batch-size 1 --learning-rate 1e-05 --weight-decay 0.0

[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `400`.

Loading weights: 100%|██████████| 162/162 [00:00<00:00, 2922.81it/s]
[transformers] VideoMAEForVideoClassification LOAD REPORT from: MCG-NJU/videomae-base-finetuned-kinetics
Key                                                            | Status     |                                                                                         
---------------------------------------------------------------+------------+-----------------------------------------------------------------------------------------
videomae.encoder.layer.{0...11}.attention.attention.q_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.v_bias     | UNEXPECTED |                                                                                         
videomae.encoder.layer.{0...11}.attention.attention.val

In [ ]:
EXPERIMENT_3 = {
    'name': 'exp3_raw50_unfreeze_lr1e-5_fc16_e50',
    'manifest': SPLIT_MANIFEST_50,
    'output_dir': MODEL_DIR / 'per_label_50_exp3_unfreeze_lr1e-5_e50',
    'epochs': 50,
    'batch_size': BATCH_SIZE,
    'learning_rate': 0.00001,
    'weight_decay': 0.05,
    'early_stopping_patience': 2,
    'frame_count': FRAME_COUNT,
    'freeze_backbone': False,
}
run_experiment(EXPERIMENT_3, enabled=RUN_TRAIN_EXP3)

In [ ]:
def print_latest_history(output_dir):
    run_dir = latest_run_dir(output_dir)
    print('run_dir:', run_dir)
    for file_name in ['run_config.json', 'training_history.csv']:
        path = run_dir / file_name
        print('##', file_name, path.exists())
        if path.suffix == '.json' and path.exists():
            print(json.dumps(json.loads(path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2)[:3000])
        elif path.exists():
            for row in read_csv(path):
                print(row)

if RUN_TRAIN_EXP1:
    print('## EXPERIMENT_1 result')
    print_latest_history(EXPERIMENT_1['output_dir'])
if RUN_TRAIN_EXP2:
    print('## EXPERIMENT_2 result')
    print_latest_history(EXPERIMENT_2['output_dir'])
if RUN_TRAIN_EXP3:
    print('## EXPERIMENT_3 result')
    print_latest_history(EXPERIMENT_3['output_dir'])


In [ ]:
# EXPORT_ANALYSIS_ARTIFACTS
import subprocess
import sys

reports_dir = PROJECT_ROOT / 'storage/vision/reports'
command = [
    sys.executable,
    'ai/vision/export_analysis_artifacts.py',
    '--root-dir', str(PROJECT_ROOT),
    '--output-dir', str(reports_dir),
]
print('$', ' '.join(command))
completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=600)
if completed.stdout:
    print(completed.stdout)
if completed.stderr:
    print(completed.stderr)
completed.check_returncode()
print('tables:', reports_dir / 'tables')
print('figures:', reports_dir / 'figures')
print('appendix:', reports_dir / 'appendix')
